In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# LOAD DATASET
# ==========================================================

# Read Heart Disease dataset
df = pd.read_csv("Heart.csv")

# Display first 5 records
print(df.head())

# ==========================================================
# DATA PREPROCESSING
# ==========================================================

# Convert target labels:
# Yes -> 1
# No  -> -1
df['AHD'] = df['AHD'].map(
    {
        'Yes': 1,
        'No': -1
    }
)

# ==========================================================
# VISUALIZATION OF DATA
# ==========================================================

# Separate positive and negative classes
class_yes = df[df['AHD'] == 1]
class_no = df[df['AHD'] == -1]

plt.figure(figsize=(8,6))

# Plot patients with heart disease
plt.scatter(
    class_yes['Age'],
    class_yes['Chol'],
    color='red',
    label='Heart Disease (+1)'
)

# Plot patients without heart disease
plt.scatter(
    class_no['Age'],
    class_no['Chol'],
    color='blue',
    label='No Heart Disease (-1)'
)

plt.xlabel("Age")
plt.ylabel("Cholesterol")

plt.title("Age vs Cholesterol")

plt.legend()

plt.show()

# ==========================================================
# FEATURE SELECTION
# ==========================================================

# Use Age and Cholesterol as input features
X = df[['Age', 'Chol']].values

# Target labels
y = df['AHD'].values

# Convert labels:
# -1 -> 0
#  1 -> 1
#
# Logistic Regression requires binary labels
y_binary = np.where(
    y == -1,
    0,
    1
)

# ==========================================================
# TRAIN-TEST SPLIT
# ==========================================================

np.random.seed(42)

# Randomly shuffle indices
indices = np.random.permutation(
    len(X)
)

# 75% Training
# 25% Testing
split = int(
    0.75 * len(X)
)

train_idx = indices[:split]
test_idx = indices[split:]

# Create train-test sets
X_train = X[train_idx]
X_test = X[test_idx]

y_train = y_binary[train_idx]
y_test = y_binary[test_idx]

# ==========================================================
# FEATURE NORMALIZATION
# ==========================================================

# Compute mean and standard deviation
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

# Standardize training data
X_train = (
    X_train - mean
) / std

# Standardize testing data
# using training statistics
X_test = (
    X_test - mean
) / std

# ==========================================================
# ADD BIAS COLUMN
# ==========================================================

def add_bias(X):

    # Add a column of ones
    return np.c_[
        np.ones(X.shape[0]),
        X
    ]

X_train = add_bias(X_train)
X_test = add_bias(X_test)

# ==========================================================
# SIGMOID FUNCTION
# ==========================================================

def sigmoid(z):

    # Logistic activation function
    return 1 / (
        1 + np.exp(-z)
    )

# ==========================================================
# LOGISTIC REGRESSION TRAINING
# ==========================================================

def train_logistic(
    X,
    y,
    lr=0.05,
    epochs=1000
):

    # m = samples
    # n = features
    m, n = X.shape

    # Initialize weights
    w = np.zeros(n)

    # Store training errors
    train_errors = []

    # Store testing errors
    test_errors = []

    for _ in range(epochs):

        # Linear combination
        z = np.dot(X, w)

        # Predicted probabilities
        predictions = sigmoid(z)

        # Compute gradient
        gradient = np.dot(
            X.T,
            (predictions - y)
        ) / m

        # Update weights
        w -= lr * gradient

        # ----------------------------------
        # Training Error
        # ----------------------------------

        train_pred = (
            predictions >= 0.5
        ).astype(int)

        train_error = np.mean(
            train_pred != y
        )

        train_errors.append(
            train_error
        )

        # ----------------------------------
        # Testing Error
        # ----------------------------------

        test_pred = sigmoid(
            np.dot(X_test, w)
        )

        test_pred_class = (
            test_pred >= 0.5
        ).astype(int)

        test_error = np.mean(
            test_pred_class != y_test
        )

        test_errors.append(
            test_error
        )

    return (
        w,
        train_errors,
        test_errors
    )

# ==========================================================
# TRAIN MODEL
# ==========================================================

weights, train_errors, test_errors = (
    train_logistic(
        X_train,
        y_train,
        lr=0.05,
        epochs=1000
    )
)

# Display learned weights
print("Learned Weights:")
print(weights)

# ==========================================================
# PLOT TRAINING AND TEST ERROR
# ==========================================================

plt.figure(figsize=(8,5))

plt.plot(
    train_errors,
    label="Training Error"
)

plt.plot(
    test_errors,
    label="Testing Error"
)

plt.xlabel("Epoch")

plt.ylabel("Classification Error")

plt.title(
    "Training vs Testing Error"
)

plt.legend()

plt.show()

# ==========================================================
# VISUALIZE DECISION BOUNDARY
# ==========================================================

# Remove bias column for plotting
X_plot = X_train[:,1:]

plt.figure(figsize=(8,6))

# Positive class
plt.scatter(
    X_plot[y_train==1][:,0],
    X_plot[y_train==1][:,1],
    color='red',
    label='Heart Disease'
)

# Negative class
plt.scatter(
    X_plot[y_train==0][:,0],
    X_plot[y_train==0][:,1],
    color='blue',
    label='No Heart Disease'
)

# Decision Boundary:
#
# w0 + w1*x1 + w2*x2 = 0
#
# x2 = -(w0 + w1*x1)/w2

x_vals = np.linspace(
    X_plot[:,0].min(),
    X_plot[:,0].max(),
    100
)

y_vals = -(
    weights[0]
    +
    weights[1] * x_vals
) / weights[2]

plt.plot(
    x_vals,
    y_vals,
    'k',
    label='Decision Boundary'
)

plt.xlabel("Normalized Age")

plt.ylabel(
    "Normalized Cholesterol"
)

plt.title(
    "Training Data with Logistic Regression Separator"
)

plt.legend()

plt.show()